# AutoPhinder walkthrough: from FIB-SEM stack to segmented autophagosomes

This notebook runs the complete AutoPhinder workflow interactively:

1. **Load** a large FIB-SEM stack lazily with Dask and read its voxel size
2. **Inspect & preprocess** the stack (trim empty slices, contrast-stretch to 8-bit)
3. **Convert IMOD annotations** into per-slice instance masks
4. **Export & curate** matched image–mask training pairs
5. **Fine-tune** micro-SAM ViT-B with an instance-segmentation decoder
6. **Segment** new data with the fine-tuned checkpoint and compare with the un-tuned model
7. **Quantify** the segmented autophagosomes

Only sections 6–7 are needed to *use* an existing AutoPhinder checkpoint; set `CHECKPOINT` in the
parameters cell and jump to section 6.

**Requirements:** the `autophinder` environment from the [installation guide](https://jurgenkriel.github.io/AutoPhinder/installation/)
plus `dask`, `tifffile` and `pandas` (`pip install -r requirements-extra.txt dask`; `zarr` optional), IMOD on your
`PATH` for section 3, and a CUDA GPU for section 5.

## 0. Parameters
Edit these paths for your data. Everything below reads from this cell.

In [ ]:
from pathlib import Path

# --- Input data -------------------------------------------------------------
SAMPLE      = "D1C2"                                   # short ID used in output file names
EM_STACK    = Path("data/D1C2_processed.ome.tif")      # FIB-SEM stack, (z, y, x)
IMOD_MODEL  = Path("data/D1C2_segmented.mod")          # IMOD model: one object per autophagosome
REPO_DIR    = Path(".")                                # AutoPhinder repository root (for scripts/)

# --- Outputs ----------------------------------------------------------------
WORK_DIR    = Path("autophinder_run")
IMAGES_DIR  = WORK_DIR / "dataset" / "images"
MASKS_DIR   = WORK_DIR / "dataset" / "masks"
RESULTS_DIR = WORK_DIR / "results"

# --- Preprocessing ----------------------------------------------------------
Z_TRIM          = (0, 0)       # slices to drop from the (start, end) of the stack, e.g. (225, 100)
FLIP_Y          = False        # flip IMOD y-coordinates if the mask overlay is upside down (see 3.3)
PERCENTILES     = (0.5, 99.5)  # contrast-stretch limits used for 8-bit conversion
CROP_MARGIN     = 128          # px of context kept around annotated objects in training tiles
MIN_TILE        = 512          # training tiles are at least this large (>= PATCH_SHAPE)
MIN_INSTANCE_PX = 25           # objects smaller than this (px) are ignored

# --- Training ---------------------------------------------------------------
MODEL_TYPE      = "vit_b"      # micro-SAM starting model
CHECKPOINT_NAME = "autophinder_model"
N_EPOCHS        = 5
ITERS_PER_EPOCH = 1000
N_OBJECTS       = 5            # objects per batch
PATCH_SHAPE     = (512, 512)
VAL_FRACTION    = 0.2

# --- Inference --------------------------------------------------------------
CHECKPOINT = WORK_DIR / "checkpoints" / CHECKPOINT_NAME / "best.pt"   # or a downloaded checkpoint
TILE_SHAPE = (1024, 1024)      # tiled inference for large slices; None to disable
HALO       = (128, 128)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import dask.array as da
import dask
import tifffile
import imageio.v3 as imageio
import matplotlib.pyplot as plt
from skimage.draw import polygon2mask
from skimage.measure import regionprops_table

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU found")

for d in (IMAGES_DIR, MASKS_DIR, RESULTS_DIR / "masks", RESULTS_DIR / "overlays"):
    d.mkdir(parents=True, exist_ok=True)

## 1. Load the FIB-SEM stack lazily

FIB-SEM stacks are often tens of GB, so never load them with `tifffile.imread()` directly. Opening the TIFF as
a Zarr store gives a lazy Dask array: only the slices you index are read from disk.

In [ ]:
def open_lazy(path):
    """Open a (possibly huge) TIFF / OME-TIFF as a lazy Dask array without reading pixel data."""
    try:
        import zarr
        z = zarr.open(tifffile.imread(path, aszarr=True), mode="r")
        return da.from_zarr(z if hasattr(z, "shape") else z[0])      # OME pyramids: level 0
    except (ImportError, ValueError, TypeError):
        # tifffile and zarr versions do not match (tifffile >= 2025 needs zarr >= 3):
        # fall back to reading one TIFF page per Dask task.
        tif = tifffile.TiffFile(path)
        series = tif.series[0].levels[0]
        page_shape = series.pages[0].shape
        read = dask.delayed(lambda i: series.pages[i].asarray())
        pages = [da.from_delayed(read(i), page_shape, series.dtype) for i in range(len(series.pages))]
        return da.stack(pages).reshape(series.shape)

em = open_lazy(EM_STACK)
if em.ndim == 4:                                          # (c, z, y, x) -> keep the EM channel
    EM_CHANNEL = 0
    em = em[EM_CHANNEL]
assert em.ndim == 3, f"Expected (z, y, x), got {em.shape}"

em = em[Z_TRIM[0]: em.shape[0] - Z_TRIM[1]]
n_z, height, width = em.shape
print(f"Stack (z, y, x): {em.shape}  dtype={em.dtype}  chunks={em.chunksize}")

### 1.1 Voxel size
Read the physical voxel size from the OME metadata so measurements can be reported in µm.

In [ ]:
import xml.etree.ElementTree as ET

def get_voxel_size(path):
    """Return (z, y, x) voxel size in µm from OME-XML, or None if absent."""
    with tifffile.TiffFile(path) as tif:
        if not tif.ome_metadata:
            return None
        root = ET.fromstring(tif.ome_metadata)
        pixels = next(el for el in root.iter() if el.tag.endswith("Pixels"))
        get = lambda k: float(pixels.attrib.get(f"PhysicalSize{k}", "nan"))
        return get("Z"), get("Y"), get("X")

VOXEL_SIZE = get_voxel_size(EM_STACK) or (0.005, 0.005, 0.005)   # fallback: 5 nm isotropic
print("Voxel size (z, y, x) µm:", VOXEL_SIZE)

## 2. Inspect and preprocess

### 2.1 Check the ends of the stack
The first and last slices of a FIB-SEM run are often blank, charged or partially milled. Look at them and set
`Z_TRIM` in the parameters cell if needed (then re-run section 1).

In [ ]:
idx = [0, n_z // 2, n_z - 1]
slices = da.compute(*[em[i] for i in idx])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, i, s in zip(axes, idx, slices):
    ax.imshow(s, cmap="gray"); ax.set_title(f"z = {i}"); ax.axis("off")
plt.tight_layout(); plt.show()

### 2.2 Contrast stretch to 8-bit
SAM expects 8-bit input. Percentile stretching is more robust than min–max scaling because a few very bright
or dark pixels (charging, curtaining) do not compress the useful range. The limits are computed **once** from
a sample of slices so every slice — training and inference — is scaled identically.

In [ ]:
sample_idx = np.linspace(0, n_z - 1, num=min(20, n_z), dtype=int)
sample = np.stack(da.compute(*[em[i, ::4, ::4] for i in sample_idx]))
P_LOW, P_HIGH = np.percentile(sample, PERCENTILES)
print(f"Intensity limits: {P_LOW:.1f} – {P_HIGH:.1f}")

def to_uint8(img, lo=P_LOW, hi=P_HIGH):
    img = np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1e-8), 0, 1)
    return (img * 255).astype(np.uint8)

raw = slices[1]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(raw, cmap="gray"); axes[0].set_title("Raw"); axes[0].axis("off")
axes[1].imshow(to_uint8(raw), cmap="gray"); axes[1].set_title("8-bit, stretched"); axes[1].axis("off")
axes[2].hist(raw.ravel(), bins=256, color="0.5"); axes[2].axvline(P_LOW, c="r"); axes[2].axvline(P_HIGH, c="r")
axes[2].set_title("Raw histogram and stretch limits")
plt.tight_layout(); plt.show()

## 3. Convert IMOD annotations into instance masks

### 3.1 Export contour points from IMOD
`model2point` (part of IMOD) writes every contour point as `object contour x y z`. Coordinates are in pixels of
the stack the model was drawn on. Run it in a terminal, or from here:

In [ ]:
POINTS = WORK_DIR / f"{SAMPLE}_points.txt"
!model2point -object -contour "{IMOD_MODEL}" "{POINTS}"

In [ ]:
pts = pd.read_csv(POINTS, sep=r"\s+", header=None, names=["object", "contour", "x", "y", "z"], comment="#")
pts["z"] = pts["z"].round().astype(int) - Z_TRIM[0]          # align with the trimmed stack
pts = pts[(pts["z"] >= 0) & (pts["z"] < n_z)]
if FLIP_Y:
    pts["y"] = height - 1 - pts["y"]

print(f"{pts['object'].nunique()} objects, {pts.groupby(['object', 'contour']).ngroups} contours, "
      f"on {pts['z'].nunique()} slices")
pts.head()

### 3.2 Rasterise contours one slice at a time
A full label volume (e.g. 1531 × 9588 × 3393) would need ~100 GB, so masks are built per slice on demand.
Each IMOD object gets its own label ID, giving an **instance** mask.

In [ ]:
slices_by_z = {z: df for z, df in pts.groupby("z")}
annotated_z = sorted(slices_by_z)

def label_slice(z, shape=(height, width)):
    """uint16 instance mask for slice z: 0 = background, IMOD object ID = autophagosome."""
    mask = np.zeros(shape, dtype=np.uint16)
    for (obj, _), contour in slices_by_z.get(z, pd.DataFrame(columns=pts.columns)).groupby(["object", "contour"]):
        if len(contour) < 3:
            continue
        poly = polygon2mask(shape, contour[["y", "x"]].to_numpy())
        mask[poly] = obj
    return mask

print(f"Annotated slices: {annotated_z[0]} – {annotated_z[-1]} ({len(annotated_z)} slices)")

### 3.3 Quality control: image, mask and overlay
The overlay must sit exactly on the autophagosome membranes. If it is mirrored vertically, set `FLIP_Y = True`;
if it is transposed (masks look rotated by 90°), the stack was opened in a different axis order than the one
used in IMOD — re-export the stack in the same orientation as the model.

In [ ]:
def show_overlay(img, mask, title="", ax=None):
    ax = ax or plt.gca()
    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(mask == 0, mask), cmap="nipy_spectral", alpha=0.5, interpolation="none")
    ax.set_title(title); ax.axis("off")

z_qc = max(annotated_z, key=lambda z: slices_by_z[z]["object"].nunique())   # busiest slice
img_qc, mask_qc = to_uint8(em[z_qc].compute()), label_slice(z_qc)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(img_qc, cmap="gray"); axes[0].set_title("FIB-SEM"); axes[0].axis("off")
axes[1].imshow(mask_qc, cmap="nipy_spectral", interpolation="none"); axes[1].set_title("Instance mask"); axes[1].axis("off")
show_overlay(img_qc, mask_qc, "Overlay", axes[2])
fig.suptitle(f"z = {z_qc}", fontsize=16); plt.tight_layout(); plt.show()

## 4. Export and curate the training dataset

### 4.1 Export image–mask tiles
For each annotated slice, a tile around the annotated objects (plus `CROP_MARGIN` of context, at least
`MIN_TILE` px wide) is written as a matched pair with identical file names. Cropping keeps the dataset small and
focuses training on regions that actually contain autophagosomes.

In [ ]:
def crop_box(mask, margin=CROP_MARGIN, min_size=MIN_TILE):
    ys, xs = np.nonzero(mask)
    box = []
    for lo, hi, n in ((ys.min(), ys.max(), mask.shape[0]), (xs.min(), xs.max(), mask.shape[1])):
        lo, hi = max(lo - margin, 0), min(hi + margin + 1, n)
        if hi - lo < min_size:                                   # grow to the minimum tile size
            pad = min_size - (hi - lo)
            lo = max(lo - pad // 2, 0); hi = min(lo + min_size, n); lo = max(hi - min_size, 0)
        box.append(slice(lo, hi))
    return tuple(box)

n_written = 0
for z in annotated_z:
    mask = label_slice(z)
    if not mask.any():
        continue
    sl = crop_box(mask)
    name = f"{SAMPLE}_frame{z + Z_TRIM[0]:04d}.tif"
    imageio.imwrite(IMAGES_DIR / name, to_uint8(em[z][sl].compute()))
    imageio.imwrite(MASKS_DIR / name, mask[sl])
    n_written += 1
print(f"Wrote {n_written} image–mask pairs to {IMAGES_DIR.parent}")

### 4.2 Curate with the repository scripts
These are the same checks as on the [Masks & dataset curation](https://jurgenkriel.github.io/AutoPhinder/dataset/)
page. Pairs exported above already match by name and shape, so the important checks are for empty or tiny masks.

In [ ]:
import sys
PY = sys.executable          # run the scripts with this notebook's Python environment
!"{PY}" "{REPO_DIR}/scripts/check_empty_masks.py" --label_dir "{MASKS_DIR}"
!"{PY}" "{REPO_DIR}/scripts/check_min_instance_size.py" --label_dir "{MASKS_DIR}" --min_instance_size {MIN_INSTANCE_PX}
!"{PY}" "{REPO_DIR}/scripts/validate_pairs_after_resize.py" --image_dir "{IMAGES_DIR}" --label_dir "{MASKS_DIR}" --min_instance_size {MIN_INSTANCE_PX}

### 4.3 Train / validation split
Neighbouring FIB-SEM slices are almost identical, so a random split leaks training data into validation. Split
by **contiguous blocks of z** instead: the last `VAL_FRACTION` of the stack is held out for validation.

In [ ]:
image_paths = sorted(IMAGES_DIR.glob("*.tif"))
label_paths = [MASKS_DIR / p.name for p in image_paths]
keep = [i for i, lp in enumerate(label_paths) if lp.exists() and imageio.imread(lp).any()]
image_paths = [str(image_paths[i]) for i in keep]
label_paths = [str(label_paths[i]) for i in keep]

n_val = max(1, int(len(image_paths) * VAL_FRACTION))
train_imgs, val_imgs = image_paths[:-n_val], image_paths[-n_val:]
train_lbls, val_lbls = label_paths[:-n_val], label_paths[-n_val:]
print(f"{len(train_imgs)} training / {len(val_imgs)} validation pairs")

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, i in zip(axes.T, np.linspace(0, len(train_imgs) - 1, 4, dtype=int)):
    img, lbl = imageio.imread(train_imgs[i]), imageio.imread(train_lbls[i])
    ax[0].imshow(img, cmap="gray"); ax[0].set_title(Path(train_imgs[i]).name); ax[0].axis("off")
    show_overlay(img, lbl, "", ax[1])
plt.tight_layout(); plt.show()

## 5. Fine-tune micro-SAM

`default_sam_loader` in image-collection mode (`raw_key=None`, lists of file paths) handles tiles of different
sizes. Most FIB-SEM patches contain no or only one autophagosome, so the `MinInstanceSampler` is set to accept
patches with **at least one** object (micro-SAM's default asks for two).

Training writes TensorBoard logs to `WORK_DIR/logs`: run `tensorboard --logdir autophinder_run/logs` to follow
`train/loss` and `train/model_iou`.

In [ ]:
import micro_sam.training as sam_training
from torch_em.data import MinInstanceSampler

sampler = MinInstanceSampler(min_num_instances=1, min_size=MIN_INSTANCE_PX)
loader_kwargs = dict(
    raw_key=None, label_key=None, patch_shape=PATCH_SHAPE,
    with_segmentation_decoder=True, sampler=sampler, batch_size=1, num_workers=4,
)
train_loader = sam_training.default_sam_loader(
    raw_paths=train_imgs, label_paths=train_lbls, shuffle=True, n_samples=ITERS_PER_EPOCH, **loader_kwargs
)
val_loader = sam_training.default_sam_loader(
    raw_paths=val_imgs, label_paths=val_lbls, shuffle=False, n_samples=max(50, ITERS_PER_EPOCH // 10),
    is_train=False, **loader_kwargs
)

In [ ]:
from torch_em.util.debug import check_loader
check_loader(train_loader, n_samples=4, plt=True)     # sanity check: every patch should show an object

In [ ]:
sam_training.train_sam(
    name=CHECKPOINT_NAME,
    model_type=MODEL_TYPE,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    n_objects_per_batch=N_OBJECTS,
    with_segmentation_decoder=True,     # trains the extra decoder used for automatic instance segmentation
    device=DEVICE,
    save_root=WORK_DIR,
)
print("Best checkpoint:", CHECKPOINT, CHECKPOINT.exists())

> **Long runs:** five epochs of 1,000 iterations take several hours on an A100. For long runs, submit the
> training as a SLURM batch job instead (see the Training page) and come back to section 6 with the checkpoint.

## 6. Segment new data

### 6.1 Load the fine-tuned model
Because the checkpoint contains the trained decoder, micro-SAM automatically uses **automatic instance
segmentation (AIS)**.

In [ ]:
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation

predictor, segmenter = get_predictor_and_segmenter(
    model_type=MODEL_TYPE, checkpoint=CHECKPOINT, device=DEVICE, is_tiled=TILE_SHAPE is not None,
)

def segment(img8):
    """Instance-segment one 8-bit slice (or crop) with AutoPhinder."""
    return automatic_instance_segmentation(
        predictor=predictor, segmenter=segmenter, input_path=img8, ndim=2,
        tile_shape=TILE_SHAPE, halo=HALO, verbose=False,
    )

### 6.2 Try it on a region of one slice
Start with a crop to check the result quickly. Pick a slice that was **not** used for training: here, from the
held-out validation block.

In [ ]:
z_test = int(Path(val_imgs[len(val_imgs) // 2]).stem.split("frame")[-1]) - Z_TRIM[0]
Y0, Y1, X0, X1 = 0, min(2048, height), 0, min(2048, width)        # change to explore other regions

test_img = to_uint8(em[z_test, Y0:Y1, X0:X1].compute())
pred = segment(test_img)
print(f"z = {z_test}: {len(np.unique(pred)) - 1} objects")

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(test_img, cmap="gray"); axes[0].set_title("Input"); axes[0].axis("off")
show_overlay(test_img, label_slice(z_test)[Y0:Y1, X0:X1], "Ground truth", axes[1])
show_overlay(test_img, pred, "AutoPhinder", axes[2])
plt.tight_layout(); plt.show()

### 6.3 Compare with the un-tuned model
The generic micro-SAM electron-microscopy model (`vit_b_em_organelles`) is a useful baseline: it finds many
organelles, while AutoPhinder should pick out autophagosomes specifically. The weights are downloaded on first use.

In [ ]:
base_predictor, base_segmenter = get_predictor_and_segmenter(model_type="vit_b_em_organelles", device=DEVICE)
base_pred = automatic_instance_segmentation(
    predictor=base_predictor, segmenter=base_segmenter, input_path=test_img, ndim=2, verbose=False,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
show_overlay(test_img, base_pred, f"vit_b_em_organelles ({len(np.unique(base_pred)) - 1} objects)", axes[0])
show_overlay(test_img, pred, f"AutoPhinder ({len(np.unique(pred)) - 1} objects)", axes[1])
plt.tight_layout(); plt.show()

> **Empty or sparse predictions?** Check the 8-bit conversion first (section 2.2). AIS thresholds can be
> relaxed via keyword arguments to `automatic_instance_segmentation`, e.g.
> `foreground_threshold=0.4, center_distance_threshold=0.6, min_size=MIN_INSTANCE_PX`. For the prompt-based
> AMG mode (`segmentation_mode="amg"` in `get_predictor_and_segmenter`), the equivalents are
> `pred_iou_thresh=0.55, stability_score_thresh=0.75`.

### 6.4 Segment the whole stack
Each slice is segmented and written to `results/masks/`, with an overlay PNG every `OVERLAY_EVERY` slices for
visual inspection. On a large stack this takes a while; run it as a batch job for production use.

In [ ]:
from tqdm.auto import tqdm

Z_RANGE       = range(0, n_z)       # restrict while testing, e.g. range(z_test - 5, z_test + 5)
OVERLAY_EVERY = 50

for z in tqdm(Z_RANGE):
    out = RESULTS_DIR / "masks" / f"{SAMPLE}_frame{z + Z_TRIM[0]:04d}.tif"
    if out.exists():
        continue                                   # resumable
    img8 = to_uint8(em[z].compute())
    seg = segment(img8).astype(np.uint16)
    imageio.imwrite(out, seg)
    if z % OVERLAY_EVERY == 0:
        fig = plt.figure(figsize=(10, 10)); show_overlay(img8, seg, f"z = {z}")
        fig.savefig(RESULTS_DIR / "overlays" / f"{out.stem}_overlay.png", dpi=120, bbox_inches="tight")
        plt.close(fig)

## 7. Quantify

Per-slice measurements in physical units. Labels are per slice; to measure 3D volumes, link instances across
slices (e.g. with `micro_sam.multi_dimensional_segmentation.automatic_3d_segmentation`, or by matching overlap
between neighbouring slices).

In [ ]:
px_area_um2 = VOXEL_SIZE[1] * VOXEL_SIZE[2]
rows = []
for f in sorted((RESULTS_DIR / "masks").glob("*.tif")):
    props = regionprops_table(imageio.imread(f), properties=("label", "area", "eccentricity", "centroid"))
    df = pd.DataFrame(props)
    df["slice"] = f.stem
    rows.append(df)

stats = pd.concat(rows, ignore_index=True)
stats = stats[stats["area"] >= MIN_INSTANCE_PX]
stats["area_um2"] = stats["area"] * px_area_um2
stats["eq_diameter_um"] = 2 * np.sqrt(stats["area_um2"] / np.pi)
stats.to_csv(RESULTS_DIR / f"{SAMPLE}_autophagosome_stats.csv", index=False)

print(f"{len(stats)} objects on {stats['slice'].nunique()} slices")
stats[["area_um2", "eq_diameter_um", "eccentricity"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
stats.groupby("slice").size().plot(ax=axes[0], color="0.3"); axes[0].set_title("Objects per slice"); axes[0].set_xticks([])
stats["eq_diameter_um"].plot.hist(bins=40, ax=axes[1], color="0.5"); axes[1].set_title("Equivalent diameter (µm)")
plt.tight_layout(); plt.show()